# BraSNAM: download dos artigos e extração dos cabeçalhos

In [4]:
%pip install -q requests beautifulsoup4 pymupdf

Note: you may need to restart the kernel to use updated packages.


## Configuração

In [3]:
from pathlib import Path

SAIDA = Path("brasnam")          # pasta de saída
SEM_PDF = False                  # True = baixa só os metadados (Parte 2 precisa dos PDFs)
ANOS = None                      # ex.: [2012, 2013] para limitar a alguns anos; None = todos
PAUSA = 1.0                      # segundos entre requisições, para não sobrecarregar o servidor

PDFS = SAIDA / "pdfs"            # onde a Parte 2 procura os PDFs (busca em subpastas)
METADADOS = SAIDA / "artigos.csv"  # CSV da Parte 1; se existir, título e ano vêm de lá

SAIDA.mkdir(exist_ok=True)

---
# Parte 1: baixar os artigos

Lê o arquivo de edições do BraSNAM na SOL, entra em cada artigo, lê as meta tags (padrão Google Scholar) e baixa o PDF.

In [ ]:
import csv
import re
import time

import requests
from bs4 import BeautifulSoup

BASE = "https://sol.sbc.org.br/index.php/brasnam"

sessao = requests.Session()
sessao.headers["User-Agent"] = "Mozilla/5.0 (pesquisa academica - levantamento bibliometrico BraSNAM)"

In [ ]:
def get(url, tentativas=3):
    for i in range(tentativas):
        try:
            r = sessao.get(url, timeout=60)
            r.raise_for_status()
            time.sleep(PAUSA)
            return r
        except requests.RequestException as e:
            print(f"  erro em {url}: {e} (tentativa {i + 1}/{tentativas})")
            time.sleep(5 * (i + 1))
    return None


def listar_edicoes():
    """Retorna lista de (url_da_edicao, titulo) a partir do arquivo de edicoes."""
    edicoes, vistos, pagina = [], set(), 1
    while True:
        url = f"{BASE}/issue/archive" + ("" if pagina == 1 else f"/{pagina}")
        r = get(url)
        if r is None:
            break
        soup = BeautifulSoup(r.text, "html.parser")
        novos = 0
        for a in soup.find_all("a", href=re.compile(r"/issue/view/\d+$")):
            href = a["href"]
            if href in vistos:
                continue
            vistos.add(href)
            titulo = a.get_text(" ", strip=True)
            if titulo:
                edicoes.append((href, titulo))
                novos += 1
        if novos == 0:
            break
        pagina += 1
    return edicoes


def listar_artigos(url_edicao):
    r = get(url_edicao)
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=re.compile(r"/article/view/\d+$")):
        if a["href"] not in links:
            links.append(a["href"])
    return links


def ler_artigo(url_artigo):
    """Le as meta tags (padrao Google Scholar) da pagina do artigo."""
    r = get(url_artigo)
    if r is None:
        return None
    soup = BeautifulSoup(r.text, "html.parser")

    def meta(nome):
        tag = soup.find("meta", attrs={"name": nome})
        return tag["content"].strip() if tag and tag.get("content") else ""

    # autores e instituicoes aparecem intercalados, na ordem do artigo
    autores = []
    for tag in soup.find_all("meta", attrs={"name": ["citation_author", "citation_author_institution"]}):
        valor = (tag.get("content") or "").strip()
        if tag["name"] == "citation_author":
            autores.append({"autor": valor, "instituicoes": []})
        elif autores and valor:
            autores[-1]["instituicoes"].append(valor)

    pdf = meta("citation_pdf_url")
    if not pdf:
        a = soup.find("a", class_=re.compile("galley")) or soup.find("a", string=re.compile("PDF", re.I))
        pdf = a["href"] if a and a.get("href") else ""
    pdf = pdf.replace("/article/view/", "/article/download/")

    return {
        "id": re.search(r"/article/view/(\d+)", url_artigo).group(1),
        "titulo": meta("citation_title"),
        "data": meta("citation_date") or meta("citation_publication_date"),
        "doi": meta("citation_doi"),
        "url": url_artigo,
        "pdf_url": pdf,
        "autores": autores,
    }


def baixar_pdf(url, destino):
    if destino.exists() and destino.stat().st_size > 0:
        return True
    r = get(url)
    if r is None or not r.content.startswith(b"%PDF"):
        print(f"  nao consegui baixar o PDF: {url}")
        return False
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(r.content)
    return True

### Executar o download

In [ ]:
print("Listando edicoes...")
edicoes = listar_edicoes()
if not edicoes:
    raise RuntimeError("Nenhuma edicao encontrada. Verifique sua conexao ou se o endereco do site mudou.")
print(f"{len(edicoes)} edicoes encontradas.")

linhas_artigos, linhas_autores = [], []
for url_ed, titulo_ed in edicoes:
    m = re.search(r"(20\d\d)", titulo_ed)
    ano = int(m.group(1)) if m else None
    if ANOS and ano not in ANOS:
        continue
    artigos = listar_artigos(url_ed)
    print(f"\n[{ano}] {titulo_ed} -> {len(artigos)} artigos")

    for i, url_art in enumerate(artigos, 1):
        art = ler_artigo(url_art)
        if art is None:
            continue
        print(f"  {i}/{len(artigos)} {art['titulo'][:70]}")

        caminho_pdf = ""
        if not SEM_PDF and art["pdf_url"]:
            destino = PDFS / str(ano) / f"{art['id']}.pdf"
            if baixar_pdf(art["pdf_url"], destino):
                caminho_pdf = str(destino)

        linhas_artigos.append({
            "id": art["id"], "ano": ano, "edicao": titulo_ed, "titulo": art["titulo"],
            "n_autores": len(art["autores"]),
            "autores": "; ".join(a["autor"] for a in art["autores"]),
            "doi": art["doi"], "url": art["url"], "pdf": caminho_pdf,
        })
        for ordem, a in enumerate(art["autores"], 1):
            linhas_autores.append({
                "id_artigo": art["id"], "ano": ano, "ordem": ordem, "autor": a["autor"],
                "instituicao": " | ".join(a["instituicoes"]), "uf": "", "regiao": "",
            })


def salvar(nome, linhas):
    if not linhas:
        return
    with open(SAIDA / nome, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
        w.writeheader()
        w.writerows(linhas)


salvar("artigos.csv", linhas_artigos)
salvar("autores.csv", linhas_autores)

sem_inst = sum(1 for l in linhas_autores if not l["instituicao"])
print(f"\nPronto: {len(linhas_artigos)} artigos, {len(linhas_autores)} autorias.")
print(f"Autorias sem instituicao nos metadados: {sem_inst} (essas precisam ser conferidas no PDF).")
print(f"Arquivos em: {SAIDA.resolve()}")

---
# Parte 2: extrair os cabeçalhos dos PDFs

Etapas:

1. Extrai o texto de cada PDF para `.txt` (`txt/<ano>/<id>.txt`).
2. Separa só o cabeçalho, até "Abstract"/"Resumo" (`cabecalhos/<ano>/<id>.txt`).
3. Identifica título, 1º e último autor, instituição e endereço de cada um.
4. Gera `cabecalhos_brasnam.csv`.

Artigos de autoria única ficam com as colunas do último autor vazias.
Pode rodar só esta parte se os PDFs já estiverem em `PDFS` (rode antes a célula de configuração).

In [ ]:
import csv
import re
import unicodedata
from pathlib import Path

try:
    import pymupdf
except ImportError:
    import fitz as pymupdf  # versoes antigas do PyMuPDF

### Limpeza do texto e leitura da 1ª página

In [ ]:
SOBRESCRITOS = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
LIGATURAS = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl", "ﬅ": "st"}
ACENTOS = {"´": "́", "`": "̀", "˜": "̃", "~": "̃", "^": "̂", "ˆ": "̂", "¨": "̈"}


def limpar(texto):
    """Corrige ligaduras e acentos soltos comuns em PDFs gerados por LaTeX antigo (ex.: 'Jo˜ao' -> 'João')."""
    for k, v in LIGATURAS.items():
        texto = texto.replace(k, v)
    texto = texto.translate(SOBRESCRITOS)
    texto = re.sub(r"([´`˜~^ˆ¨])\s?([A-Za-zı])",
                   lambda m: unicodedata.normalize("NFC", m.group(2).replace("ı", "i") + ACENTOS[m.group(1)]), texto)
    texto = re.sub(r"([A-Za-zı])\s?([´`˜ˆ¨])(?=\s|$)",  # acento depois da letra
                   lambda m: unicodedata.normalize("NFC", m.group(1).replace("ı", "i") + ACENTOS[m.group(2)]), texto)
    texto = re.sub(r"([cC])\s?¸", lambda m: "ç" if m.group(1) == "c" else "Ç", texto)
    texto = texto.replace("ı", "i")
    return unicodedata.normalize("NFC", re.sub(r"[ \t]+", " ", texto)).strip()


def linhas_primeira_pagina(pagina):
    """Reconstroi as linhas da 1a pagina com tamanho de fonte, juntando sobrescritos a linha certa."""
    flags = pymupdf.TEXTFLAGS_DICT & ~getattr(pymupdf, "TEXT_PRESERVE_LIGATURES", 0)
    spans = []
    for bloco in pagina.get_text("dict", flags=flags)["blocks"]:
        for linha in bloco.get("lines", []):
            for s in linha["spans"]:
                if s["text"].strip():
                    spans.append(s)
    spans.sort(key=lambda s: (s["bbox"][1], s["bbox"][0]))

    linhas = []  # cada linha: lista de spans
    for s in spans:
        y0, y1 = s["bbox"][1], s["bbox"][3]
        destino = None
        for ln in linhas[-4:]:
            ly0 = min(x["bbox"][1] for x in ln)
            ly1 = max(x["bbox"][3] for x in ln)
            sobreposicao = min(y1, ly1) - max(y0, ly0)
            if sobreposicao > 0.5 * min(y1 - y0, ly1 - ly0):
                destino = ln
                break
        if destino is None:
            linhas.append([s])
        else:
            destino.append(s)

    resultado = []
    for ln in linhas:
        ln.sort(key=lambda s: s["bbox"][0])
        tamanho = max(s["size"] for s in ln)
        centro = max(ln, key=lambda s: s["size"])
        cy = (centro["bbox"][1] + centro["bbox"][3]) / 2
        texto, anterior = "", None
        for s in ln:
            sobrescrito = (s["flags"] & 1) or (
                s["size"] < 0.85 * tamanho and (s["bbox"][1] + s["bbox"][3]) / 2 < cy - 0.1 * tamanho)
            if anterior is not None and not sobrescrito and s["bbox"][0] - anterior["bbox"][2] > 0.15 * tamanho:
                texto += " "
            elif anterior is not None and sobrescrito and texto and not texto.endswith(" "):
                pass  # sobrescrito fica colado no nome: "Silva1"
            texto += s["text"]
            if sobrescrito:
                texto += " "
            anterior = s
        texto = limpar(texto)
        if texto:
            resultado.append({"texto": texto, "tamanho": round(tamanho, 1), "y": ln[0]["bbox"][1]})
    resultado.sort(key=lambda l: l["y"])
    return resultado

### Cabeçalho: título, autores e afiliações

In [ ]:
FIM_CABECALHO = re.compile(r"^\s*(abstract|resumo|resumen)\b", re.I)
SECAO = re.compile(r"^\s*1\.?\s+(introdu|introduction)", re.I)
EMAIL = re.compile(r"@|\{[^}]*\}\s*@?|\bemail\b|\be-mail\b", re.I)
MARCA_AFILIACAO = re.compile(r"^\s*(\d{1,2}|[*†‡])\s*(?=[A-Za-zÀ-ÿ(])(.*)$")

INST_FORTE = re.compile(
    r"\b(Universidade|Universit(y|é|ät|à|at)|Universidad|Univ\.|Instituto Federal|Centro Federal|CEFET|"
    r"Faculdades?|Centro Universit[aá]rio|College|SENAI|Embrapa|Fiocruz|Petrobras|INPE|IMPA|LNCC|RNP|"
    r"Escola Superior|Escola Polit[eé]cnica|Polit[eé]cnic|Institut[oe]? Nacional|Funda[cç][aã]o)\b", re.I)
UNIVERSIDADE = re.compile(  # tem prioridade sobre escola/instituto/departamento interno
    r"\b(Universidade|Universit(y|é|ät|à|at)|Universidad|Univ\.|Instituto Federal|Centro Federal|CEFET|"
    r"Centro Universit[aá]rio)\b", re.I)
INST_FRACA = re.compile(
    r"\b(Instituto|Institute|Institut|Centro|Center|Centre|Escola|School|Departamento|Department|Dept|Faculty|"
    r"Programa|Program|Graduate|P[oó]s-Gradua[cç][aã]o|Laborat[oó]rio|Laboratory|Lab|N[uú]cleo|Grupo|Group|"
    r"Coordena[cç][aã]o|Foundation|Empresa|Company|Inc|Ltda|Secretaria|Minist[eé]rio|Ag[eê]ncia|Agency|"
    r"Research|Pesquisa|Unidade|Divis[aã]o|Division|Reitoria)\b", re.I)


def separar_cabecalho(linhas):
    """Retorna (linhas_do_cabecalho, achou_fim). O cabecalho comeca no titulo (maior fonte)."""
    fim = None
    for i, l in enumerate(linhas):
        if FIM_CABECALHO.match(l["texto"]) or SECAO.match(l["texto"]):
            fim = i
            break
    corpo = linhas[:fim] if fim is not None else linhas[:25]
    if not corpo:
        return [], fim is not None
    maior = max(l["tamanho"] for l in corpo)
    inicio = next(i for i, l in enumerate(corpo) if l["tamanho"] >= 0.92 * maior)
    return corpo[inicio:], fim is not None


def separar_titulo(cab):
    maior = cab[0]["tamanho"]
    n = 0
    while n < len(cab) and cab[n]["tamanho"] >= 0.92 * maior:
        n += 1
    titulo = re.sub(r"-\s+(?=[a-zà-ÿ])", "", " ".join(l["texto"] for l in cab[:n]))
    return titulo, cab[n:]


def eh_inicio_afiliacao(texto):
    return bool(EMAIL.search(texto) or MARCA_AFILIACAO.match(texto)
                or INST_FORTE.search(texto) or INST_FRACA.search(texto))


def parse_autores(texto):
    """'Ana Silva1,2, Bruno Souza2 e Carla Lima1' -> [('Ana Silva', ['1','2']), ...]"""
    texto = re.sub(r"\s+(and|e|&)\s+", ", ", texto)
    autores = []
    for seg in texto.split(","):
        seg = seg.strip(" ;")
        if not seg:
            continue
        if re.fullmatch(r"[\d\s*†‡]+", seg):
            if autores:
                autores[-1][1].extend(re.findall(r"\d+", seg))
            continue
        m = re.match(r"^(.*?)[\s\d*†‡]*$", seg)
        nome = m.group(1).strip()
        marcas = re.findall(r"\d+", seg[len(m.group(1)):])
        if nome:
            autores.append((nome, marcas))
    return autores


def parse_afiliacoes(linhas):
    """Agrupa as linhas de afiliacao por numero ('1', '2'...). Sem numeros, tudo vai para '_'."""
    blocos, atual = {}, None
    for t in linhas:
        if EMAIL.search(t):
            continue
        m = MARCA_AFILIACAO.match(t)
        if m:
            atual = m.group(1)
            blocos.setdefault(atual, []).append(m.group(2).strip())
            continue
        if atual is None:
            atual = "_"
            blocos[atual] = []
        blocos[atual].append(t)
    return blocos


SIGLA = re.compile(r"\(?(?=[A-Za-z0-9-]{3,}\)?$)[A-Z][a-z0-9-]*[A-Z][A-Za-z0-9-]*\)?")  # UFAM, UnB, PUC-Rio
PAIS_BR = re.compile(r"\b(Brasil|Brazil|BRASIL|BRAZIL)\b")


def parece_endereco(t):
    s = sem_acento(t)
    return bool(re.search(r"(?<![A-Za-z])(" + "|".join(UFS) + r")(?![A-Za-z])", t)
                or re.search(r"\d{5}-?\d{3}", t) or PAIS_BR.search(t) or EXTERIOR.search(t)
                or any(re.search(rf"\b{re.escape(c)}\b", s) for c in CIDADES))


def juntar_linhas_quebradas(linhas_bloco):
    """'Universidade Federal de Minas' + 'Gerais' -> uma linha so."""
    juntas = []
    for t in linhas_bloco:
        continuacao = (juntas and not re.search(r"\s[–—-]\s|[,;]", t) and not parece_endereco(t)
                       and not INST_FORTE.match(t) and not INST_FRACA.match(t)
                       and (INST_FORTE.search(juntas[-1]) or INST_FRACA.search(juntas[-1]))
                       and not parece_endereco(juntas[-1].split(" – ")[-1]))
        if continuacao:
            juntas[-1] = f"{juntas[-1]} {t}"
        else:
            juntas.append(t)
    return juntas


def dividir_instituicao_endereco(linhas_bloco):
    """Separa os pedacos de instituicao dos pedacos de endereco."""
    pedacos = []
    for t in juntar_linhas_quebradas(linhas_bloco):
        for p in re.split(r"\s+[–—-]\s+|\s*[,;]\s*", t):
            p = p.strip(" .")
            if p:
                pedacos.append(p)
    inst, ender = [], []
    for p in pedacos:
        nucleo = p.strip("()")
        sigla = (SIGLA.fullmatch(p) and nucleo not in UFS and not PAIS_BR.search(p)
                 and not EXTERIOR.fullmatch(nucleo))
        if INST_FORTE.search(p) or INST_FRACA.search(p) or sigla:
            inst.append(p)
        else:
            ender.append(p)
    principal = (next((p for p in inst if UNIVERSIDADE.search(p)), None)
                 or next((p for p in inst if INST_FORTE.search(p)), None) or (inst[0] if inst else ""))
    # junta a sigla que vem logo depois: "Universidade Federal do Amazonas (UFAM)"
    if principal:
        i = inst.index(principal)
        if i + 1 < len(inst) and SIGLA.fullmatch(inst[i + 1]) and "(" not in principal:
            principal = f"{principal} ({inst[i + 1].strip('()')})"
    return principal, ", ".join(ender)

### UF e região

In [ ]:
UFS = {"AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR",
       "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"}
REGIAO = {**dict.fromkeys(["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Norte"),
          **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
          **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
          **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
          **dict.fromkeys(["PR", "RS", "SC"], "Sul")}
ESTADOS = {"acre": "AC", "alagoas": "AL", "amapa": "AP", "amazonas": "AM", "bahia": "BA", "ceara": "CE",
           "distrito federal": "DF", "espirito santo": "ES", "goias": "GO", "maranhao": "MA",
           "mato grosso do sul": "MS", "mato grosso": "MT", "minas gerais": "MG", "paraiba": "PB",
           "parana": "PR", "pernambuco": "PE", "piaui": "PI", "rio de janeiro": "RJ",
           "rio grande do norte": "RN", "rio grande do sul": "RS", "rondonia": "RO", "roraima": "RR",
           "santa catarina": "SC", "sao paulo": "SP", "sergipe": "SE", "tocantins": "TO"}
CIDADES = {"rio branco": "AC", "maceio": "AL", "macapa": "AP", "manaus": "AM", "salvador": "BA",
           "fortaleza": "CE", "brasilia": "DF", "vitoria": "ES", "goiania": "GO", "sao luis": "MA",
           "cuiaba": "MT", "campo grande": "MS", "belo horizonte": "MG", "belem": "PA", "joao pessoa": "PB",
           "curitiba": "PR", "recife": "PE", "teresina": "PI", "natal": "RN", "porto alegre": "RS",
           "porto velho": "RO", "boa vista": "RR", "florianopolis": "SC", "aracaju": "SE", "palmas": "TO",
           "niteroi": "RJ", "petropolis": "RJ", "seropedica": "RJ", "nova iguacu": "RJ",
           "campinas": "SP", "sao carlos": "SP", "santo andre": "SP", "sao bernardo do campo": "SP",
           "ribeirao preto": "SP", "sorocaba": "SP", "sao jose dos campos": "SP", "guarulhos": "SP",
           "bauru": "SP", "presidente prudente": "SP", "sao jose do rio preto": "SP",
           "juiz de fora": "MG", "uberlandia": "MG", "vicosa": "MG", "lavras": "MG", "itajuba": "MG",
           "ouro preto": "MG", "sao joao del-rei": "MG", "sao joao del rei": "MG", "montes claros": "MG",
           "divinopolis": "MG", "uberaba": "MG", "formiga": "MG",
           "campina grande": "PB", "vitoria da conquista": "BA", "feira de santana": "BA", "ilheus": "BA",
           "jequie": "BA", "camacari": "BA", "santarem": "PA", "maraba": "PA", "londrina": "PR",
           "maringa": "PR", "ponta grossa": "PR", "cascavel": "PR", "pelotas": "RS", "santa maria": "RS",
           "alegrete": "RS", "bage": "RS", "caxias do sul": "RS", "sao leopoldo": "RS", "rio grande": "RS",
           "joinville": "SC", "blumenau": "SC", "itajai": "SC", "mossoro": "RN", "caruaru": "PE",
           "garanhuns": "PE", "parnaiba": "PI", "picos": "PI", "sobral": "CE", "quixada": "CE",
           "crato": "CE", "juazeiro do norte": "CE", "imperatriz": "MA", "anapolis": "GO",
           "rio verde": "GO", "dourados": "MS", "rondonopolis": "MT", "itacoatiara": "AM",
           "parintins": "AM", "colatina": "ES", "serra": "ES", "vila velha": "ES"}
EXTERIOR = re.compile(
    r"\b(USA|U\.S\.A|United States|EUA|Estados Unidos|Portugal|France|Fran[cç]a|Spain|Espanha|Argentina|"
    r"Uruguay|Uruguai|Chile|Germany|Alemanha|UK|United Kingdom|Reino Unido|England|Italy|It[aá]lia|"
    r"Canada|Canad[aá]|M[eé]xico|Colombia|Col[oô]mbia|Peru|Netherlands|Holanda|Switzerland|Su[ií][cç]a|"
    r"Taiwan|China|Japan|Jap[aã]o|Australia|Austr[aá]lia|Ireland|Irlanda|Belgium|B[eé]lgica|Sweden|"
    r"Austria|India|Qatar|Israel|Finland|Norway|Denmark|Paraguay|Paraguai|Ecuador|Equador|Venezuela|"
    r"Bol[ií]via|Greece|Poland|Singapore|Korea|Coreia|Luxembourg|Luxemburgo)\b")


def sem_acento(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn").lower()


def descobrir_uf(endereco, bloco_inteiro):
    if EXTERIOR.search(bloco_inteiro) and not re.search(r"\b(Brasil|Brazil)\b", bloco_inteiro, re.I):
        return "", "Exterior"
    m = re.search(r"(?<![A-Za-z])(" + "|".join(UFS) + r")(?![A-Za-z])", endereco)
    if m:
        return m.group(1), REGIAO[m.group(1)]
    # "Pará" so com acento, para nao confundir com a preposicao "para"
    if re.search(r"\bPará\b|\b(do|of) Para\b", bloco_inteiro):
        return "PA", "Norte"
    for texto in (sem_acento(endereco), sem_acento(bloco_inteiro)):
        for tabela in (CIDADES, ESTADOS):
            for nome in sorted(tabela, key=len, reverse=True):
                if re.search(rf"\b{re.escape(nome)}\b", texto):
                    return tabela[nome], REGIAO[tabela[nome]]
    return "", ""

### Processamento de um PDF

In [ ]:
def processar(pdf, pasta_txt, pasta_cab, ano, titulos_meta):
    doc = pymupdf.open(pdf)
    texto_completo = "\n\n".join(limpar(p.get_text()) for p in doc)
    (pasta_txt / ano).mkdir(parents=True, exist_ok=True)
    (pasta_txt / ano / f"{pdf.stem}.txt").write_text(texto_completo, encoding="utf-8")

    linhas = linhas_primeira_pagina(doc[0]) if len(doc) else []
    doc.close()
    cab, achou_fim = separar_cabecalho(linhas)
    (pasta_cab / ano).mkdir(parents=True, exist_ok=True)
    (pasta_cab / ano / f"{pdf.stem}.txt").write_text("\n".join(l["texto"] for l in cab), encoding="utf-8")

    avisos = []
    if not achou_fim:
        avisos.append("nao achei Abstract/Resumo")
    if not cab:
        return None, ["cabecalho vazio (PDF escaneado?)"]

    titulo, resto = separar_titulo(cab)
    textos = [l["texto"] for l in resto]
    i = 0
    while i < len(textos) and not eh_inicio_afiliacao(textos[i]):
        i += 1
    autores = parse_autores(" , ".join(textos[:i]))
    blocos = parse_afiliacoes(textos[i:])

    if not autores:
        avisos.append("nenhum autor encontrado")
    if len(blocos) > 1 and any(not marcas for _, marcas in (autores[:1] + autores[-1:])):
        avisos.append("autor sem numero de afiliacao")

    def afiliacao(n):
        if n >= len(autores):
            return "", "", "", "", ""
        nome, marcas = autores[n]
        chave = next((m for m in marcas if m in blocos), None)
        if chave is None:
            chave = "_" if "_" in blocos else (next(iter(blocos)) if blocos else None)
        if chave is None:
            return nome, "", "", "", ""
        inst, ender = dividir_instituicao_endereco(blocos[chave])
        uf, regiao = descobrir_uf(ender, " ".join(blocos[chave]))
        return nome, inst, ender, uf, regiao

    a1 = afiliacao(0)
    # ultimo autor so existe com 2 ou mais autores; em autoria unica fica vazio
    au = afiliacao(len(autores) - 1) if len(autores) >= 2 else ("", "", "", "", "")
    if autores and not a1[1]:
        avisos.append("instituicao do 1o autor vazia")
    if len(autores) > 1 and not au[1]:
        avisos.append("instituicao do ultimo autor vazia")
    for rotulo, a in (("1o", a1), ("ultimo", au)):
        if a[0] and not a[3] and a[4] != "Exterior":
            avisos.append(f"UF do {rotulo} autor nao identificada")

    return {
        "Nome do artigo": titulos_meta.get(pdf.stem) or titulo,
        "Nome do primeiro autor": a1[0],
        "Nome do último autor": au[0],
        "Instituição do 1º autor": a1[1],
        "Instituição do último autor": au[1],
        "Endereço da instituição do 1º autor": a1[2],
        "Endereço da instituição do último autor": au[2],
        "UF do 1º autor": a1[3],
        "Região do 1º autor": a1[4],
        "UF do último autor": au[3],
        "Região do último autor": au[4],
        "Total de autores": len(autores),
        "Ano": ano,
        "Arquivo": str(pdf),
    }, avisos

### Executar a extração

In [ ]:
pdfs = sorted(Path(PDFS).rglob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"Nenhum PDF encontrado em {PDFS}")

titulos_meta, anos_meta = {}, {}
if Path(METADADOS).exists():
    with open(METADADOS, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            if r.get("titulo"):
                titulos_meta[r["id"]] = r["titulo"]
            if r.get("ano"):
                anos_meta[r["id"]] = r["ano"]

linhas_csv = []
for n, pdf in enumerate(pdfs, 1):
    # ano vem da subpasta (pdfs/<ano>/) ou, com os PDFs todos numa pasta so, do artigos.csv
    ano = pdf.parent.name if pdf.parent.name.isdigit() else anos_meta.get(pdf.stem, "sem_ano")
    try:
        linha, avisos = processar(pdf, SAIDA / "txt", SAIDA / "cabecalhos", ano, titulos_meta)
    except Exception as e:  # PDF corrompido etc.
        linha, avisos = None, [f"erro: {e}"]
    if linha is None:
        linha = {"Nome do artigo": titulos_meta.get(pdf.stem, ""), "Ano": ano, "Arquivo": str(pdf)}
    linha["Verificar"] = "; ".join(avisos)
    linhas_csv.append(linha)
    print(f"{n}/{len(pdfs)} {pdf.name}" + (f"  [verificar: {linha['Verificar']}]" if avisos else ""))

colunas = ["Nome do artigo", "Nome do primeiro autor", "Nome do último autor",
           "Instituição do 1º autor", "Instituição do último autor",
           "Endereço da instituição do 1º autor", "Endereço da instituição do último autor",
           "UF do 1º autor", "Região do 1º autor", "UF do último autor", "Região do último autor",
           "Total de autores", "Ano", "Arquivo", "Verificar"]
# garante campo vazio de verdade: sem espacos, "None", "nan" etc.,
# e colunas do ultimo autor sempre vazias quando o artigo tem autoria unica
colunas_2o = [c for c in colunas if "último" in c]
for linha in linhas_csv:
    for c in colunas:
        v = linha.get(c)
        v = "" if v is None else str(v).strip()
        linha[c] = "" if v.lower() in ("none", "nan", "null", "n/a") else v
    if linha["Total de autores"] in ("", "0", "1"):
        for c in colunas_2o:
            linha[c] = ""

destino = SAIDA / "cabecalhos_brasnam.csv"
with open(destino, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=colunas, restval="")
    w.writeheader()
    w.writerows(linhas_csv)

marcados = sum(1 for l in linhas_csv if l["Verificar"])
print(f"\nPronto: {len(linhas_csv)} artigos em {destino}")
print(f"{marcados} marcados para conferencia manual (coluna 'Verificar').")